# Basketball-51 Baseline: Shot Classification (3pt / 2pt / Free Throw)
### Based on Shakya et al. — 3D ResNet-50 & I3D Backbones

This notebook implements a baseline video classification pipeline for the **Basketball-51** dataset.  
We train on **16-frame clips** at **224×224** resolution using either a **3D ResNet-50** or **I3D** backbone, following the paper's experimental protocol.

**Key specs:**
- Cross-Entropy loss, SGD (momentum 0.9, weight-decay 1e-4)
- LR 0.001, stepped ×0.1 every 10 epochs, 50 epochs total
- Mixed-precision training (AMP) for GPU efficiency
- Evaluation: Top-1 Accuracy + Confusion Matrix

## 1. Setup Environment & Mount Google Drive

In [ ]:
# ── Install dependencies ──
!pip install -q pytorchvideo opencv-python-headless scikit-learn seaborn matplotlib tqdm

# ── Mount Google Drive ──
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Set dataset root (adjust to your Drive folder structure) ──
DATASET_ROOT = '/content/drive/MyDrive/Basketball-51'

# Verify directory exists and list class subfolders
assert os.path.isdir(DATASET_ROOT), f"Dataset not found at {DATASET_ROOT}"
class_dirs = sorted([
    d for d in os.listdir(DATASET_ROOT)
    if os.path.isdir(os.path.join(DATASET_ROOT, d))
])
print(f"Found {len(class_dirs)} class folders: {class_dirs}")
for d in class_dirs:
    n = len(os.listdir(os.path.join(DATASET_ROOT, d)))
    print(f"  {d}: {n} files")

## 2. Configure GPU Device & Hyperparameters

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True  # faster convolutions when input sizes are fixed

# ── Device ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected — training will be extremely slow!")

# ── Hyperparameters ──
CFG = {
    'BATCH_SIZE':     8,
    'NUM_FRAMES':    16,
    'FRAME_SIZE':   224,
    'NUM_CLASSES':    3,
    'EPOCHS':        50,
    'LR':         1e-3,
    'MOMENTUM':    0.9,
    'WEIGHT_DECAY':1e-4,
    'LR_STEP_SIZE': 10,
    'LR_GAMMA':    0.1,
    'NUM_WORKERS':   4,
    'PIN_MEMORY': True,
}

CLASS_NAMES = ['2pt', '3pt', 'FreeThrow']   # alphabetical — adjust if your folders differ
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

print("\nConfiguration:")
for k, v in CFG.items():
    print(f"  {k}: {v}")
print(f"\nClasses: {CLASS_TO_IDX}")

## 3. Video Preprocessing & Augmentation Transforms

Training transforms include **temporal jittering** (random start offset when sampling 16 frames),
random horizontal flip, and ImageNet normalisation.  
Validation uses deterministic centre-sampling only.

In [ ]:
import torchvision.transforms as T

# ImageNet statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def get_spatial_transforms(is_train: bool):
    """Return spatial transforms applied independently to each frame."""
    if is_train:
        return T.Compose([
            T.ToPILImage(),
            T.Resize((CFG['FRAME_SIZE'], CFG['FRAME_SIZE'])),
            T.RandomHorizontalFlip(p=0.5),
            T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return T.Compose([
            T.ToPILImage(),
            T.Resize((CFG['FRAME_SIZE'], CFG['FRAME_SIZE'])),
            T.ToTensor(),
            T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])


print("Transforms defined ✓")

## 4. Custom Video Dataset Class

Reads videos from a folder-per-class layout using OpenCV.  
**Temporal jittering**: during training, the start frame of the 16-frame window is randomly offset; during validation, frames are sampled from the centre of the video. Videos shorter than 16 frames are looped.

In [ ]:
import cv2
from torch.utils.data import Dataset, DataLoader


class Basketball51Dataset(Dataset):
    """
    Custom Dataset for Basketball-51 shot classification.
    Loads video clips from a folder-per-class directory layout.
    Returns tensors of shape (C, T, H, W) — channels-first, time second.
    """

    VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

    def __init__(self, video_paths: list, labels: list, is_train: bool = True):
        """
        Args:
            video_paths: list of absolute paths to video files.
            labels: list of integer labels corresponding to each video.
            is_train: if True, apply temporal jittering + augmentation.
        """
        self.video_paths = video_paths
        self.labels = labels
        self.is_train = is_train
        self.num_frames = CFG['NUM_FRAMES']
        self.spatial_transform = get_spatial_transforms(is_train)

    def __len__(self):
        return len(self.video_paths)

    def _load_video_frames(self, video_path: str):
        """Read video with OpenCV and sample `num_frames` consecutive frames."""
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Cannot open video: {video_path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # ── Determine start frame ──
        if total_frames <= self.num_frames:
            start = 0
        elif self.is_train:
            # Temporal jittering: random start offset
            start = random.randint(0, total_frames - self.num_frames)
        else:
            # Centre crop temporally
            start = (total_frames - self.num_frames) // 2

        cap.set(cv2.CAP_PROP_POS_FRAMES, start)

        frames = []
        for _ in range(self.num_frames):
            ret, frame = cap.read()
            if not ret:
                # Loop back to beginning if video is shorter than num_frames
                cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
                ret, frame = cap.read()
                if not ret:
                    break
            # OpenCV returns BGR → convert to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()

        # Pad with last frame if still short
        while len(frames) < self.num_frames:
            frames.append(frames[-1])

        return frames  # list of H×W×3 numpy arrays

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = self._load_video_frames(video_path)

        # Apply spatial transforms to each frame → list of (C, H, W) tensors
        frames_t = [self.spatial_transform(f) for f in frames]

        # Stack along temporal dim → (C, T, H, W)
        clip = torch.stack(frames_t, dim=1)  # (3, 16, 224, 224)

        return clip, label


# ── Helper: scan dataset directory and build file lists ──
def scan_dataset(root: str, class_to_idx: dict):
    """Walk the folder-per-class directory and return (paths, labels)."""
    paths, labels = [], []
    for class_name, idx in class_to_idx.items():
        class_dir = os.path.join(root, class_name)
        if not os.path.isdir(class_dir):
            print(f"  [WARN] Folder not found for class '{class_name}': {class_dir}")
            continue
        for fname in sorted(os.listdir(class_dir)):
            ext = os.path.splitext(fname)[1].lower()
            if ext in Basketball51Dataset.VIDEO_EXTENSIONS:
                paths.append(os.path.join(class_dir, fname))
                labels.append(idx)
    return paths, labels


print("Basketball51Dataset class defined ✓")

## 5. Create Data Loaders (80 / 20 Stratified Split)

In [ ]:
from sklearn.model_selection import train_test_split
from collections import Counter

# ── Scan full dataset ──
all_paths, all_labels = scan_dataset(DATASET_ROOT, CLASS_TO_IDX)
print(f"Total videos found: {len(all_paths)}")
print("Class distribution:", {CLASS_NAMES[k]: v for k, v in sorted(Counter(all_labels).items())})

# ── Stratified train / val split ──
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.20,
    stratify=all_labels,
    random_state=SEED,
)

print(f"\nTrain: {len(train_paths)}  |  Val: {len(val_paths)}")
print("Train distribution:", {CLASS_NAMES[k]: v for k, v in sorted(Counter(train_labels).items())})
print("Val   distribution:", {CLASS_NAMES[k]: v for k, v in sorted(Counter(val_labels).items())})

# ── Instantiate Datasets ──
train_dataset = Basketball51Dataset(train_paths, train_labels, is_train=True)
val_dataset   = Basketball51Dataset(val_paths,   val_labels,   is_train=False)

# ── DataLoaders ──
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=True,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=CFG['PIN_MEMORY'],
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CFG['BATCH_SIZE'],
    shuffle=False,
    num_workers=CFG['NUM_WORKERS'],
    pin_memory=CFG['PIN_MEMORY'],
    drop_last=False,
)

print(f"\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# ── Quick sanity check ──
sample_clip, sample_label = next(iter(train_loader))
print(f"Sample batch — clip shape: {sample_clip.shape}, labels: {sample_label}")

## 6. 3D ResNet-50 Backbone

A full 3D ResNet-50 implementation using `nn.Conv3d` with bottleneck blocks and residual skip connections.  
Layer configuration: **[3, 4, 6, 3]** blocks — matching the standard ResNet-50 depth.

In [ ]:
class Bottleneck3D(nn.Module):
    """3D Bottleneck block for ResNet-50."""
    expansion = 4

    def __init__(self, in_channels, mid_channels, stride=1, downsample=None,
                 temporal_stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, mid_channels, kernel_size=1, bias=False)
        self.bn1   = nn.BatchNorm3d(mid_channels)

        self.conv2 = nn.Conv3d(
            mid_channels, mid_channels,
            kernel_size=3,
            stride=(temporal_stride, stride, stride),
            padding=1, bias=False,
        )
        self.bn2 = nn.BatchNorm3d(mid_channels)

        self.conv3 = nn.Conv3d(mid_channels, mid_channels * self.expansion,
                               kernel_size=1, bias=False)
        self.bn3   = nn.BatchNorm3d(mid_channels * self.expansion)

        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        return self.relu(out)


class ResNet3D50(nn.Module):
    """
    3D ResNet-50 for video classification.
    Input:  (B, 3, T, H, W) — e.g. (B, 3, 16, 224, 224)
    Output: (B, num_classes)
    """

    def __init__(self, num_classes: int = 3):
        super().__init__()
        self.in_channels = 64

        # Stem: 7×7×7 conv
        self.conv1 = nn.Conv3d(3, 64, kernel_size=(3, 7, 7),
                               stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
        self.bn1   = nn.BatchNorm3d(64)
        self.relu  = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2),
                                    padding=(0, 1, 1))

        # Residual layers — [3, 4, 6, 3]
        self.layer1 = self._make_layer(64,  3, stride=1, temporal_stride=1)
        self.layer2 = self._make_layer(128, 4, stride=2, temporal_stride=2)
        self.layer3 = self._make_layer(256, 6, stride=2, temporal_stride=2)
        self.layer4 = self._make_layer(512, 3, stride=2, temporal_stride=2)

        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(512 * Bottleneck3D.expansion, num_classes)

        # Weight initialisation
        self._init_weights()

    def _make_layer(self, mid_channels, num_blocks, stride, temporal_stride):
        downsample = None
        out_channels = mid_channels * Bottleneck3D.expansion

        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv3d(self.in_channels, out_channels, kernel_size=1,
                          stride=(temporal_stride, stride, stride), bias=False),
                nn.BatchNorm3d(out_channels),
            )

        layers = [Bottleneck3D(self.in_channels, mid_channels,
                               stride=stride, downsample=downsample,
                               temporal_stride=temporal_stride)]
        self.in_channels = out_channels
        for _ in range(1, num_blocks):
            layers.append(Bottleneck3D(self.in_channels, mid_channels))

        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


print("ResNet3D50 defined ✓")

## 7. I3D (Inflated 3D ConvNet) Backbone

An Inflated Inception–based architecture. Each inception module has parallel 3D convolution branches (1×1×1, 3×3×3, 5×5×5 factorised as two 3×3×3) plus a max-pool branch.  
A utility is provided to inflate pretrained 2D ImageNet weights into 3D by repeating along the temporal axis.

In [ ]:
class InceptionModule3D(nn.Module):
    """Single 3D Inception module with 4 parallel branches."""

    def __init__(self, in_ch, ch1x1, ch3x3_reduce, ch3x3,
                 ch5x5_reduce, ch5x5, pool_proj):
        super().__init__()

        # Branch 1: 1×1×1
        self.branch1 = nn.Sequential(
            nn.Conv3d(in_ch, ch1x1, kernel_size=1, bias=False),
            nn.BatchNorm3d(ch1x1),
            nn.ReLU(inplace=True),
        )

        # Branch 2: 1×1×1 → 3×3×3
        self.branch2 = nn.Sequential(
            nn.Conv3d(in_ch, ch3x3_reduce, kernel_size=1, bias=False),
            nn.BatchNorm3d(ch3x3_reduce),
            nn.ReLU(inplace=True),
            nn.Conv3d(ch3x3_reduce, ch3x3, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(ch3x3),
            nn.ReLU(inplace=True),
        )

        # Branch 3: 1×1×1 → 3×3×3 → 3×3×3 (factorised 5×5×5)
        self.branch3 = nn.Sequential(
            nn.Conv3d(in_ch, ch5x5_reduce, kernel_size=1, bias=False),
            nn.BatchNorm3d(ch5x5_reduce),
            nn.ReLU(inplace=True),
            nn.Conv3d(ch5x5_reduce, ch5x5, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(ch5x5),
            nn.ReLU(inplace=True),
            nn.Conv3d(ch5x5, ch5x5, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(ch5x5),
            nn.ReLU(inplace=True),
        )

        # Branch 4: 3×3×3 max-pool → 1×1×1 projection
        self.branch4 = nn.Sequential(
            nn.MaxPool3d(kernel_size=3, stride=1, padding=1),
            nn.Conv3d(in_ch, pool_proj, kernel_size=1, bias=False),
            nn.BatchNorm3d(pool_proj),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return torch.cat([
            self.branch1(x), self.branch2(x),
            self.branch3(x), self.branch4(x),
        ], dim=1)


class I3D(nn.Module):
    """
    Inflated 3D ConvNet (simplified Inception-based).
    Input:  (B, 3, T, H, W)
    Output: (B, num_classes)
    """

    def __init__(self, num_classes: int = 3):
        super().__init__()

        # ── Stem ──
        self.stem = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(7, 7, 7), stride=(2, 2, 2),
                      padding=(3, 3, 3), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2),
                         padding=(0, 1, 1)),

            nn.Conv3d(64, 64, kernel_size=1, bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),

            nn.Conv3d(64, 192, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm3d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2),
                         padding=(0, 1, 1)),
        )

        # ── Inception blocks ──
        #                       in   1x1   3r  3x3   5r  5x5  pool
        self.inc3a = InceptionModule3D(192,  64, 96, 128, 16,  32,  32)   # out=256
        self.inc3b = InceptionModule3D(256, 128,128, 192, 32,  96,  64)   # out=480
        self.pool3 = nn.MaxPool3d(kernel_size=(2, 2, 2), stride=2)

        self.inc4a = InceptionModule3D(480, 192, 96, 208, 16,  48,  64)   # out=512
        self.inc4b = InceptionModule3D(512, 160,112, 224, 24,  64,  64)   # out=512
        self.inc4c = InceptionModule3D(512, 128,128, 256, 24,  64,  64)   # out=512
        self.inc4d = InceptionModule3D(512, 112,144, 288, 32,  64,  64)   # out=528
        self.inc4e = InceptionModule3D(528, 256,160, 320, 32, 128, 128)   # out=832
        self.pool4 = nn.MaxPool3d(kernel_size=2, stride=2)

        self.inc5a = InceptionModule3D(832, 256,160, 320, 32, 128, 128)   # out=832
        self.inc5b = InceptionModule3D(832, 384,192, 384, 48, 128, 128)   # out=1024

        # ── Head ──
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(1024, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)

        x = self.inc3a(x)
        x = self.inc3b(x)
        x = self.pool3(x)

        x = self.inc4a(x)
        x = self.inc4b(x)
        x = self.inc4c(x)
        x = self.inc4d(x)
        x = self.inc4e(x)
        x = self.pool4(x)

        x = self.inc5a(x)
        x = self.inc5b(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


# ── Utility: inflate 2D weights → 3D ──
def inflate_2d_to_3d(state_dict_2d, temporal_kernel_size=3):
    """
    Take a 2D state dict (e.g. from torchvision Inception/ResNet) and
    repeat each (out, in, H, W) conv weight along a new temporal axis,
    rescaling by 1/T so activations stay in the same range.
    """
    state_dict_3d = {}
    for k, v in state_dict_2d.items():
        if v.ndim == 4:  # Conv2d weight → Conv3d
            t = temporal_kernel_size
            # (out, in, H, W) → (out, in, T, H, W)
            v_3d = v.unsqueeze(2).repeat(1, 1, t, 1, 1) / t
            state_dict_3d[k] = v_3d
        else:
            state_dict_3d[k] = v
    return state_dict_3d


print("I3D defined ✓")

## 8. Model Selection & GPU Transfer

Choose the backbone below. Both models accept `(B, 3, 16, 224, 224)` input and output `(B, 3)` logits.

In [ ]:
# ────────────────────────────────────────────
#  Choose backbone: 'resnet3d50' or 'i3d'
# ────────────────────────────────────────────
MODEL_NAME = 'resnet3d50'   # <── change to 'i3d' to switch

if MODEL_NAME == 'resnet3d50':
    model = ResNet3D50(num_classes=CFG['NUM_CLASSES'])
elif MODEL_NAME == 'i3d':
    model = I3D(num_classes=CFG['NUM_CLASSES'])
else:
    raise ValueError(f"Unknown model: {MODEL_NAME}")

model = model.to(device)

# ── Parameter count ──
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {MODEL_NAME}")
print(f"  Total params:     {total_params:,}")
print(f"  Trainable params: {trainable_params:,}")

# ── Sanity check: forward pass with dummy data ──
dummy = torch.randn(2, 3, CFG['NUM_FRAMES'], CFG['FRAME_SIZE'], CFG['FRAME_SIZE']).to(device)
with torch.no_grad():
    out = model(dummy)
print(f"  Dummy forward — input: {dummy.shape}  →  output: {out.shape}")
assert out.shape == (2, CFG['NUM_CLASSES']), f"Unexpected output shape: {out.shape}"
del dummy, out
torch.cuda.empty_cache()
print("  Sanity check passed ✓")

## 9. Loss Function, Optimizer & LR Scheduler

- **Loss:** Cross-Entropy
- **Optimizer:** SGD, momentum 0.9, weight-decay 1e-4
- **Scheduler:** StepLR — reduce LR ×0.1 every 10 epochs
- **AMP:** GradScaler for mixed-precision training on GPU

In [ ]:
criterion = nn.CrossEntropyLoss().to(device)

optimizer = optim.SGD(
    model.parameters(),
    lr=CFG['LR'],
    momentum=CFG['MOMENTUM'],
    weight_decay=CFG['WEIGHT_DECAY'],
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=CFG['LR_STEP_SIZE'],
    gamma=CFG['LR_GAMMA'],
)

# Mixed-precision scaler
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

print("Loss, optimizer, scheduler, and AMP scaler configured ✓")

## 10. Training Loop

Uses `torch.cuda.amp.autocast()` for FP16 forward passes and `GradScaler` for stable mixed-precision back-propagation.

In [ ]:
from tqdm.auto import tqdm


def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler):
    """
    Train for one epoch.
    Returns: (avg_loss, accuracy %)
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc='  Train', leave=False)
    for clips, labels in pbar:
        clips  = clips.to(device, non_blocking=True)   # (B, 3, 16, 224, 224)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
            outputs = model(clips)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # ── Metrics ──
        running_loss += loss.item() * clips.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total   += labels.size(0)

        pbar.set_postfix(loss=f"{loss.item():.4f}",
                         acc=f"{100.*correct/total:.1f}%")

    epoch_loss = running_loss / total
    epoch_acc  = 100. * correct / total
    return epoch_loss, epoch_acc


print("train_one_epoch() defined ✓")

## 11. Validation Loop (Top-1 Accuracy)

Runs inference in `torch.no_grad()` mode and collects all predictions + ground truths for confusion matrix generation.

In [ ]:
def validate(model, dataloader, criterion, device):
    """
    Run validation pass.
    Returns: (avg_loss, top1_accuracy %, all_preds, all_targets)
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    pbar = tqdm(dataloader, desc='  Val  ', leave=False)
    with torch.no_grad():
        for clips, labels in pbar:
            clips  = clips.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(clips)
                loss = criterion(outputs, labels)

            running_loss += loss.item() * clips.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}",
                             acc=f"{100.*correct/total:.1f}%")

    epoch_loss = running_loss / total
    epoch_acc  = 100. * correct / total
    return epoch_loss, epoch_acc, all_preds, all_targets


print("validate() defined ✓")

## 12. Execute Training (50 Epochs)

The main loop calls `train_one_epoch` → `validate` → `scheduler.step()` for each epoch.  
Best model weights (by val accuracy) are checkpointed to Google Drive.

In [ ]:
import time

# ── Checkpoint path ──
SAVE_DIR = '/content/drive/MyDrive/Basketball-51/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(SAVE_DIR, f'best_{MODEL_NAME}.pth')

# ── History ──
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   [],
    'lr': [],
}

best_val_acc = 0.0
patience_counter = 0
PATIENCE = 15  # optional early stopping patience (set to EPOCHS to disable)

print(f"Training {MODEL_NAME} for {CFG['EPOCHS']} epochs …\n")
total_start = time.time()

for epoch in range(1, CFG['EPOCHS'] + 1):
    epoch_start = time.time()

    # ── Train ──
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device, scaler,
    )

    # ── Validate ──
    val_loss, val_acc, val_preds, val_targets = validate(
        model, val_loader, criterion, device,
    )

    # ── Scheduler step ──
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    # ── Log ──
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    elapsed = time.time() - epoch_start
    print(f"Epoch {epoch:>3}/{CFG['EPOCHS']}  │  "
          f"Train Loss: {train_loss:.4f}  Acc: {train_acc:5.1f}%  │  "
          f"Val Loss: {val_loss:.4f}  Acc: {val_acc:5.1f}%  │  "
          f"LR: {current_lr:.1e}  │  {elapsed:.0f}s")

    # ── Checkpoint best model ──
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'cfg': CFG,
        }, BEST_MODEL_PATH)
        print(f"  ↳ Best model saved (val_acc={val_acc:.1f}%)")
    else:
        patience_counter += 1

    # ── Early stopping ──
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} epochs "
              f"(no improvement for {PATIENCE} epochs).")
        break

total_time = time.time() - total_start
print(f"\n{'='*60}")
print(f"Training complete in {total_time/60:.1f} minutes")
print(f"Best validation accuracy: {best_val_acc:.1f}%")

## 13. Training & Validation Curves

In [ ]:
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['train_loss']) + 1)
best_epoch = int(np.argmax(history['val_acc'])) + 1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss ──
ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   label='Val Loss')
ax1.axvline(best_epoch, color='gray', ls='--', alpha=0.5, label=f'Best epoch ({best_epoch})')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Accuracy ──
ax2.plot(epochs_range, history['train_acc'], label='Train Acc')
ax2.plot(epochs_range, history['val_acc'],   label='Val Acc')
ax2.axvline(best_epoch, color='gray', ls='--', alpha=0.5, label=f'Best epoch ({best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, f'{MODEL_NAME}_curves.png'), dpi=150)
plt.show()
print(f"Figure saved to {SAVE_DIR}")

## 14. Confusion Matrix (Best Model)

Loads the best checkpoint, runs a full validation pass, and visualises the confusion matrix.

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

# ── Load best checkpoint ──
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Loaded best model from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.1f}%)")

# ── Full validation pass ──
_, final_acc, all_preds, all_targets = validate(model, val_loader, criterion, device)
print(f"Final Top-1 Accuracy: {final_acc:.1f}%")

# ── Confusion Matrix ──
cm = confusion_matrix(all_targets, all_preds)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    ax=ax, linewidths=0.5,
)

# Add percentage annotations
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j + 0.5, i + 0.72, f"({cm_pct[i, j]:.0f}%)",
                ha='center', va='center', fontsize=9, color='gray')

ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title(f'Confusion Matrix — {MODEL_NAME} (Top-1 Acc: {final_acc:.1f}%)', fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(SAVE_DIR, f'{MODEL_NAME}_confusion_matrix.png'), dpi=150)
plt.show()

## 15. Per-Class Precision, Recall & F1-Score

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    all_targets, all_preds,
    target_names=CLASS_NAMES,
    digits=3,
)
print(f"Classification Report — {MODEL_NAME}\n")
print(report)

# ── Identify most/least confused pairs ──
np.fill_diagonal(cm.copy(), 0)
cm_off = cm.copy()
np.fill_diagonal(cm_off, 0)
max_conf = np.unravel_index(cm_off.argmax(), cm_off.shape)
print(f"Most confused pair: True={CLASS_NAMES[max_conf[0]]} → "
      f"Predicted={CLASS_NAMES[max_conf[1]]} ({cm_off[max_conf]} samples)")

print("\nDone! Best model checkpoint and figures saved to Google Drive.")